# Are the translator-study translations independent of each other?

The translator study (33 translations of 14 languages) compares translations of the same language.
The two Bosnian translations turned out to share 58% of their verses word for word, so at least one
pair is a revision rather than an independent translation. This notebook checks every pair:
- the share of verses that are identical, after keeping letters only;
- the containment of 12-character letter shingles, the measure used for the Tanzil check.

It then recomputes the within-language spread using only pairs that pass the near-copy filter
applied to eBible (fewer than half of shared verses identical). That makes the comparison with the
Bible fair.

CPU, about 25 minutes, almost all of it downloading from QuranEnc (cached, like the translator
notebook). The notebook asks for one file, `fertility_by_translation.csv`, which is in
`anon_repo/results/translator_variance/`.

In [ ]:
import os, json, time, html, itertools, unicodedata
import numpy as np, pandas as pd, requests
OUT = "results_independence"; CACHE = "quranenc_cache"
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
NAME = "fertility_by_translation.csv"
try:
    import google.colab  # noqa: F401
    from google.colab import files
    while not os.path.exists(NAME):
        print(f"Upload {NAME} (anon_repo/results/translator_variance/)")
        up = files.upload(); got = list(up)[0] if up else None
        if got and got != NAME and got.startswith("fertility_by_translation"): os.replace(got, NAME)
except ImportError:
    pass
FT = pd.read_csv(NAME)
KEYS = sorted(FT.translation_key)
print(len(KEYS), "translations in the study:", KEYS)

SURA_URL = "https://quranenc.com/api/v1/translation/sura/{key}/{sura}"
def fetch(key, retries=3):
    path = f"{CACHE}/{key}.json"
    if os.path.exists(path): return pd.read_json(path)
    rows = []
    for sura in range(1, 115):
        for a in range(retries):
            try:
                r = requests.get(SURA_URL.format(key=key, sura=sura), timeout=60); r.raise_for_status()
                rows.extend(r.json()["result"]); break
            except Exception as e:
                if a == retries - 1: print(f"  ! {key} sura {sura}: {e}"); return None
                time.sleep(2 ** a)
        time.sleep(0.15)
    df = pd.DataFrame(rows); df.to_json(path, orient="records", force_ascii=False); return df

def letters(t):
    t = unicodedata.normalize("NFKC", html.unescape(str(t))).casefold()
    return "".join(ch for ch in t if unicodedata.category(ch)[0] in "LM")

TEXT = {}
for k in KEYS:
    df = fetch(k)
    if df is None: continue
    df = df[["sura", "aya", "translation"]].astype({"sura": int, "aya": int}).drop_duplicates(["sura", "aya"])
    TEXT[k] = {(r.sura, r.aya): letters(r.translation) for r in df.itertuples()}
    print(f"{k}: {len(TEXT[k])} verses")

In [ ]:
def shingles(d, K=12):
    out = set()
    for s in d.values(): out.update(hash(s[i:i + K]) for i in range(len(s) - K + 1))
    return out
SH = {k: shingles(v) for k, v in TEXT.items()}
rows = []
lang = dict(zip(FT.translation_key, FT.language))
for l, g in FT.groupby("language"):
    for a, b in itertools.combinations(sorted(k for k in g.translation_key if k in TEXT), 2):
        both = set(TEXT[a]) & set(TEXT[b])
        same = sum(TEXT[a][v] == TEXT[b][v] for v in both)
        inter = len(SH[a] & SH[b])
        rows.append({"language": l, "a": a, "b": b, "verses": len(both), "identical_verses": same / len(both),
                     "containment_a_in_b": inter / len(SH[a]), "containment_b_in_a": inter / len(SH[b])})
PAIRS = pd.DataFrame(rows)
PAIRS["near_copy"] = PAIRS.identical_verses >= 0.5          # the eBible filter
PAIRS.round(3).to_csv(f"{OUT}/pairs.csv", index=False)
print(PAIRS.round(3).sort_values("identical_verses", ascending=False).to_string(index=False))

In [ ]:
F = FT.set_index("translation_key")
def summary(pairs, label):
    out = []
    for m in ["gpt4o_o200k_tpv", "bytes_per_verse"]:
        per_lang = []
        for l, g in pairs.groupby("language"):
            r = [max(F.loc[a, m], F.loc[b, m]) / min(F.loc[a, m], F.loc[b, m]) for a, b in zip(g.a, g.b)]
            per_lang.append(np.median(r))
        out.append({"pairs": label, "measure": m, "languages": len(per_lang), "n_pairs": len(pairs),
                    "median_pair_ratio": round(float(np.median(per_lang)), 3)})
    return out
S = pd.DataFrame(summary(PAIRS, "all pairs") + summary(PAIRS[~PAIRS.near_copy], "independent pairs only"))
S.to_csv(f"{OUT}/within_language_summary.csv", index=False)
print(S.to_string(index=False))
print("\nnear-copy pairs:", PAIRS[PAIRS.near_copy][["a", "b"]].values.tolist())
print("Bible (eBible, near-copies removed): median pair ratio 1.067 GPT-4o, 1.053 bytes")
json.dump({"near_copies": PAIRS[PAIRS.near_copy][["a", "b", "identical_verses"]].round(3).to_dict(orient="records"),
           "summary": S.to_dict(orient="records")}, open(f"{OUT}/summary.json", "w"), indent=1)
import shutil; shutil.make_archive("results_independence", "zip", OUT)
try:
    from google.colab import files; files.download("results_independence.zip")
except Exception:
    print("written results_independence.zip")